
# Qwen3-14B + LoRA + AWQ + vLLM Pipeline

本ノートブックは Colab 上で以下の順番で実行することを想定しています。
1. **ヘルパー初期化**: pip ラッパやパッケージリストを定義し、リポジトリを clone。
2. **AWQ フェーズのセットアップ**: 量子化に必要な依存をインストール（llmcompressor 優先 / `compressed-tensors==0.12.2`）。
3. **LoRA マージ & AWQ 量子化**: 量子化 artefact を生成（セル3）。
4. **AWQ フェーズのクリーンアップ**: AWQ 専用依存をアンインストールして vLLM 用に環境を戻す。
5. **vLLM フェーズのセットアップ**: vLLM 依存を再インストール（`compressed-tensors==0.11.0` / `setuptools==79.0.1`）。
6. **vLLM での推論とメトリクス計測**。
7. **対話デモとクリーンアップ**。

> ℹ️ **Colab メモ**
> - AWQ セルと vLLM セルの間で必ずクリーンアップセルを実行してください。フェーズをまたぐときはランタイムを再起動するとより安全です。
> - 保存先を切り替えたい場合は `LLMLAB_DRIVE_DATA_ROOT` または `LLMLAB_WORKSPACE_DATA_ROOT` を設定します。
> - Google Drive を手動でマウント済みなら `LLMLAB_SKIP_DRIVE_MOUNT=1` でマウント処理をスキップできます。

In [ ]:
# === 1. Colab utility setup ===
import os
import shlex
import sys
from typing import Sequence

try:
    from IPython import get_ipython
except ImportError:  # pragma: no cover
    get_ipython = None  # type: ignore


def _require_ipython():
    ip = get_ipython() if callable(get_ipython) else None
    if ip is None:
        raise RuntimeError("IPython environment is required. Please run on Google Colab.")
    return ip


def clone_repo(url: str, target: str, branch: str | None = None) -> None:
    if os.path.exists(target):
        print("Reusing existing repository:", target)
        return
    ip = _require_ipython()
    if branch:
        print(f"Cloning repository: {url} (branch={branch})")
        ip.system(f"git clone --branch {branch} --single-branch {url} {target}")
    else:
        print("Cloning repository:", url)
        ip.system(f"git clone {url} {target}")


def _run_pip(arguments: Sequence[str]) -> None:
    ip = _require_ipython()
    command = " ".join(shlex.quote(str(arg)) for arg in arguments)
    print("pip", command)
    ip.run_line_magic("pip", command)


def pip_install(
    packages: Sequence[str],
    *,
    index_url: str | None = None,
    extra_index_url: str | None = None,
    upgrade: bool = True,
    extra_args: Sequence[str] | None = None,
) -> None:
    items = [pkg for pkg in packages if pkg]
    if not items:
        return
    args = ["install"]
    if upgrade:
        args.append("--upgrade")
    if index_url:
        args.extend(["--index-url", index_url])
    if extra_index_url:
        args.extend(["--extra-index-url", extra_index_url])
    if extra_args:
        args.extend(extra_args)
    args.extend(items)
    _run_pip(args)


def pip_uninstall(packages: Sequence[str]) -> None:
    items = [pkg for pkg in packages if pkg]
    if not items:
        return
    args = ["uninstall", "-y"]
    args.extend(items)
    _run_pip(args)


REPO_URL = "https://github.com/fouga1221/llm-lab2.git"
DEFAULT_REPO_DIR = "/content/llm-lab2" if "google.colab" in sys.modules else os.path.abspath("..")
REPO_DIR = DEFAULT_REPO_DIR  # Change here if you want a different clone path.
REPO_BRANCH = "main2"  # Set to None to use the remote default branch.

IS_COLAB = "google.colab" in sys.modules
DEFAULT_TORCH_INDEX = "https://download.pytorch.org/whl/cu126" if IS_COLAB else None
TORCH_INDEX_URL = os.environ.get("PYTORCH_WHL_INDEX_URL", DEFAULT_TORCH_INDEX)
TORCH_EXTRA_INDEX_URL = os.environ.get("PYTORCH_EXTRA_INDEX_URL", "https://pypi.org/simple")

BASE_BOOTSTRAP = [
    "pip>=24.2",
    "setuptools==79.0.1",
    "wheel>=0.44.0",
    "packaging>=24.2",
    "jedi>=0.19.1",
]

TORCH_PACKAGES = [
    "torch==2.8.0",
    "torchvision==0.23.0",
    "torchaudio==2.8.0",
]

AWQ_NUMPY_SPEC = ["numpy==2.1.3"]
AWQ_RUNTIME_PACKAGES = [
    "transformers==4.56.2",
    "peft==0.17.1",
    "accelerate==1.10.1",
    "safetensors>=0.4.2",
    "openai>=1.99.1,<2.0.0",
]
AWQ_DATA_PACKAGES = [
    "datasets==4.0.0",
    "pyarrow==19.0.1",
]
AWQ_TELEMETRY_PACKAGES = [
    "opentelemetry-api==1.37.0",
    "opentelemetry-sdk==1.37.0",
    "opentelemetry-exporter-otlp==1.37.0",
    "opentelemetry-exporter-otlp-proto-grpc==1.37.0",
    "opentelemetry-exporter-otlp-proto-http==1.37.0",
    "opentelemetry-exporter-otlp-proto-common==1.37.0",
    "opentelemetry-proto==1.37.0",
    "opentelemetry-semantic-conventions==0.58b0",
]
AWQ_LLMCOMPRESSOR_PACKAGES = [
    "llmcompressor==0.8.1",
    "compressed-tensors==0.12.2",
]

AWQ_CLEANUP_TARGETS = [
    "llmcompressor",
    "compressed-tensors",
    "datasets",
    "pyarrow",
    "opentelemetry-exporter-otlp",
    "opentelemetry-exporter-otlp-proto-grpc",
    "opentelemetry-exporter-otlp-proto-http",
    "opentelemetry-exporter-otlp-proto-common",
    "opentelemetry-proto",
    "opentelemetry-sdk",
    "opentelemetry-semantic-conventions",
    "opentelemetry-api",
]

VLLM_SUPPORT_PACKAGES = [
    "ninja",
    "cmake>=3.26.1",
    "numba==0.61.2",
    "llvmlite==0.44.0",
    "xformers==0.0.32.post1",
    "ray[cgraph]==2.50.1",
    "msgspec==0.19.0",
    "gguf>=0.17.1",
    "datasets>=4.0.0,<5.0.0",
]

VLLM_CORE_PACKAGES = [
    "compressed-tensors==0.11.0",
    "vllm==0.11.0",
]

clone_repo(REPO_URL, REPO_DIR, branch=REPO_BRANCH)
print("Package helper setup completed.")



## 1A. AWQフェーズ用依存のインストール

AWQ 量子化を実行する前に本セルを一度だけ実行し、llmcompressor 周辺の依存関係を整えます。別作業後に戻ってきた場合はランタイム再起動後に再度実行してください。

In [ ]:
# === 1A. Install dependencies for AWQ ===
pip_install(BASE_BOOTSTRAP)
pip_install(TORCH_PACKAGES, index_url=TORCH_INDEX_URL, extra_index_url=TORCH_EXTRA_INDEX_URL)
pip_install(AWQ_NUMPY_SPEC)
pip_install(AWQ_RUNTIME_PACKAGES)
pip_install(AWQ_TELEMETRY_PACKAGES)
pip_install(AWQ_DATA_PACKAGES)
pip_install(AWQ_LLMCOMPRESSOR_PACKAGES)
print("AWQ dependency stack ready.")

In [ ]:
# === 2. Path and model configuration ===
import os
import sys
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules

COLAB_DATA_ROOT_DEFAULT = Path("/content/drive/MyDrive/ProjectForte/llm-lab-save")
COLAB_WORKSPACE_ROOT_DEFAULT = Path("/content/llm-lab-save")
COLAB_DATA_ROOT = Path(os.environ.get("LLMLAB_DRIVE_DATA_ROOT", COLAB_DATA_ROOT_DEFAULT))
workspace_override = os.environ.get("LLMLAB_WORKSPACE_DATA_ROOT")
COLAB_WORKSPACE_ROOT = Path(workspace_override) if workspace_override else COLAB_WORKSPACE_ROOT_DEFAULT

if IS_COLAB:
    from google.colab import drive  # type: ignore

    skip_mount = os.environ.get("LLMLAB_SKIP_DRIVE_MOUNT", "").lower() in {"1", "true", "yes"}
    if skip_mount:
        print("Skipping Google Drive mount as requested by LLMLAB_SKIP_DRIVE_MOUNT.")
    else:
        drive.mount("/content/drive", force_remount=False)

    if workspace_override:
        COLAB_WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
        DATA_ROOT = COLAB_WORKSPACE_ROOT
    else:
        COLAB_DATA_ROOT.mkdir(parents=True, exist_ok=True)
        if COLAB_WORKSPACE_ROOT.exists() or COLAB_WORKSPACE_ROOT.is_symlink():
            print("Reusing existing data root:", COLAB_WORKSPACE_ROOT)
        else:
            COLAB_WORKSPACE_ROOT.symlink_to(COLAB_DATA_ROOT, target_is_directory=True)
        DATA_ROOT = COLAB_WORKSPACE_ROOT
else:
    local_override = os.environ.get("LLMLAB_LOCAL_DATA_ROOT")
    DATA_ROOT = Path(local_override) if local_override else Path.cwd() / "save"
    DATA_ROOT.mkdir(parents=True, exist_ok=True)

if not DATA_ROOT.exists():
    DATA_ROOT.mkdir(parents=True, exist_ok=True)

REPO_ROOT = Path(REPO_DIR).resolve()

CONFIG = {
    "BASE_MODEL_NAME": "Qwen/Qwen3-14B",
    "LORA_DIR": DATA_ROOT / "models" / "adapters" / "qwen3-14b-lora-ojousama",
    "MERGED_OUTPUT_DIR": DATA_ROOT / "artifacts" / "merged_qwen3_14b_ojousama",
    "AWQ_OUTPUT_DIR": DATA_ROOT / "artifacts" / "awq_qwen3_14b_ojousama",
    "EXTERNAL_MERGED_DIR": None,
    "EXTERNAL_AWQ_DIR": None,
    "PERFORM_LORA_MERGE": True,
    "PERFORM_AWQ_QUANT": True,
    "AWQ_CONFIG": {
        "num_bits": 4,
        "group_size": 128,
        "max_seq_len": 512,
    },
    "AWQ_CALIBRATION_SAMPLES": [
        " ".join(
            (
                "Calibration sample block A sentence {i}. "
                "This synthetic paragraph maintains diversity with instructions, status reports, "
                "and contextual hints about quantization workflows for large language models. "
                "It references AWQ calibration routines, tensor inspection, and fallback recovery paths "
                "while enumerating checkpoints such as step {i} and step {next_i}. "
                "By repeating rich vocabulary—metrics, regulators, assistants, deploy scripts—we emulate "
                "the varied prompts typically fed into tokenizer pipelines."
            ).format(i=i, next_i=i + 1)
            for i in range(1, 161)
        ),
        " ".join(
            (
                "Calibration sample block B sentence {i}. "
                "The passage describes data preprocessing, dataset audits, experiment tracking, "
                "and mixed precision validation suited for llm-compressor instrumentation. "
                "It highlights retry logic, watchdog timers, safety valves, and telemetry exports "
                "to ensure the token stream exceeds the default 512 token threshold. "
                "Each iteration cites scenario labels {i}a and {next_i}b to broaden linguistic shape."
            ).format(i=i, next_i=i + 1)
            for i in range(1, 161)
        ),
    ],
}
VLLM_OPTIONS = {
    "tensor_parallel_size": 1,
    "dtype": "auto",
    "max_model_len": 4096,
    "gpu_memory_utilization": 0.9,
    "download_dir": DATA_ROOT / "cache" / "huggingface",
}



In [ ]:
# === 3. LoRA merge on CPU and AWQ quantisation ===
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import gc
import json
import tempfile
from pathlib import Path
from textwrap import dedent

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from llmcompressor.entrypoints.oneshot import oneshot as run_oneshot

from src.llmlab.utils.awq import prepare_awq_calib_data


def _cleanup_cuda_cache() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def build_awq_recipe(num_bits: int, group_size: int) -> str:
    return dedent(
        f"""
        modifiers:
          - AWQModifier:
              config_groups:
                group_0:
                  targets:
                    - Linear
                  weights:
                    num_bits: {num_bits}
                    type: int
                    symmetric: false
                    strategy: group
                    group_size: {group_size}
        """
    ).strip()


def write_calibration_json(samples, directory: Path) -> Path:
    directory.mkdir(parents=True, exist_ok=True)
    file_path = directory / "calibration.json"
    payload = [{"text": sample} for sample in samples]
    file_path.write_text(
        "\n".join(json.dumps(item, ensure_ascii=False) for item in payload) + "\n",
        encoding="utf-8",
    )
    return file_path


def analyse_calibration_samples(samples, tokenizer) -> None:
    data, text_column = prepare_awq_calib_data(samples)
    print(f"Calibration payload type: {type(data).__name__} / text column hint: {text_column}")
    total_tokens = 0
    if isinstance(data, list) and data:
        first = data[0]
        if isinstance(first, str):
            for idx, sample in enumerate(data):
                tokens = tokenizer.encode(sample, add_special_tokens=False)
                print(f"Sample {idx}: {len(tokens)} token(s)")
                total_tokens += len(tokens)
        elif isinstance(first, list):
            for idx, token_ids in enumerate(data):
                print(f"Sample {idx}: {len(token_ids)} token(s)")
                total_tokens += len(token_ids)
    print(f"Total calibration tokens: {total_tokens}")


def ensure_awq_config_files(model_dir: Path, awq_cfg: dict[str, int | bool]) -> None:
    if not model_dir.exists():
        return
    target_quant_config = model_dir / "quant_config.json"
    target_quantize_config = model_dir / "quantize_config.json"
    target_awq_config = model_dir / "awq_config.json"

    if target_quant_config.exists() or target_quantize_config.exists():
        if not target_awq_config.exists():
            # Prefer quant_config if available, else quantize_config.
            source = target_quant_config if target_quant_config.exists() else target_quantize_config
            target_awq_config.write_text(source.read_text(encoding="utf-8"), encoding="utf-8")
            print(f"Created {target_awq_config.name} from {source.name}.")
        return

    # Fallback: synthesise config from notebook settings.
    synthetic = {
        "quant_method": "awq",
        "w_bit": awq_cfg.get("num_bits", 4),
        "bits": awq_cfg.get("num_bits", 4),
        "group_size": awq_cfg.get("group_size", 128),
        "q_group_size": awq_cfg.get("group_size", 128),
        "zero_point": awq_cfg.get("zero_point", True),
        "modules_to_not_convert": [],
    }
    text = json.dumps(synthetic, indent=2, ensure_ascii=False)
    target_quant_config.write_text(text + "
", encoding="utf-8")
    target_awq_config.write_text(text + "
", encoding="utf-8")
    print(f"Created synthetic AWQ config files in {model_dir}.")


base_model_name = CONFIG["BASE_MODEL_NAME"]
lora_dir = CONFIG["LORA_DIR"]
merged_dir = CONFIG["MERGED_OUTPUT_DIR"]
awq_dir = CONFIG["AWQ_OUTPUT_DIR"]
external_merged_dir = CONFIG.get("EXTERNAL_MERGED_DIR")
external_awq_dir = CONFIG.get("EXTERNAL_AWQ_DIR")

if external_merged_dir:
    external_merged_dir = Path(external_merged_dir)
if external_awq_dir:
    external_awq_dir = Path(external_awq_dir)

source_model_for_awq = base_model_name

if CONFIG["PERFORM_LORA_MERGE"]:
    if not lora_dir.exists():
        raise FileNotFoundError(f"LoRA directory not found: {lora_dir}")
    print("Merging LoRA into the base model on CPU...")
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        device_map={"": "cpu"},
        torch_dtype=torch.float32,
        trust_remote_code=True,
        low_cpu_mem_usage=False,
    )
    lora_model = PeftModel.from_pretrained(base_model, str(lora_dir))
    merged_model = lora_model.merge_and_unload()
    merged_model.save_pretrained(merged_dir, safe_tensors=True)
    tokenizer_for_merge = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
    tokenizer_for_merge.save_pretrained(merged_dir)
    del base_model, lora_model, merged_model, tokenizer_for_merge
    _cleanup_cuda_cache()
    source_model_for_awq = str(merged_dir)
else:
    for candidate in [external_merged_dir, merged_dir]:
        if candidate and candidate.exists() and any(candidate.iterdir()):
            print("Using prebuilt merged weights:", candidate)
            source_model_for_awq = str(candidate)
            break
    else:
        print("No merged weights found; falling back to the base model.")

awq_recipe_cfg = CONFIG["AWQ_CONFIG"]
awq_samples = CONFIG["AWQ_CALIBRATION_SAMPLES"]
num_bits = awq_recipe_cfg.get("num_bits", 4)
group_size = awq_recipe_cfg.get("group_size", 128)
max_seq_len = awq_recipe_cfg.get("max_seq_len", 512)

if CONFIG["PERFORM_AWQ_QUANT"]:
    target_awq_dir = external_awq_dir or awq_dir
    target_awq_dir.mkdir(parents=True, exist_ok=True)
    existing_weights = list(target_awq_dir.glob("*.safetensors"))
    if existing_weights:
        print("Using existing llm-compressor artefacts:", target_awq_dir)
    else:
        print("Quantising with llm-compressor AWQ (this may take some time)...")
        tokenizer = AutoTokenizer.from_pretrained(source_model_for_awq, trust_remote_code=True)
        analyse_calibration_samples(awq_samples, tokenizer)
        recipe_yaml = build_awq_recipe(num_bits=num_bits, group_size=group_size)
        with tempfile.TemporaryDirectory() as tmpdir:
            dataset_dir = Path(tmpdir) / "dataset"
            write_calibration_json(awq_samples, dataset_dir)
            run_oneshot(
                model=source_model_for_awq,
                tokenizer=source_model_for_awq,
                trust_remote_code_model=True,
                recipe=recipe_yaml,
                dataset="json",
                dataset_path=str(dataset_dir),
                text_column="text",
                num_calibration_samples=len(awq_samples),
                max_seq_length=max_seq_len,
                output_dir=str(target_awq_dir),
            )
        _cleanup_cuda_cache()
    VLLM_MODEL_PATH = str(target_awq_dir)
    VLLM_QUANTIZATION = "awq"
else:
    candidate_awq_dirs = [external_awq_dir, awq_dir]
    selected_awq = None
    for candidate in candidate_awq_dirs:
        if candidate and candidate.exists() and any(candidate.glob("*.safetensors")):
            selected_awq = candidate
            break
    if selected_awq:
        print("Using prebuilt llm-compressor artefacts:", selected_awq)
        VLLM_MODEL_PATH = str(selected_awq)
        VLLM_QUANTIZATION = "awq"
    else:
        print("Quantised artefacts not found; using the non-quantised source model.")
        VLLM_MODEL_PATH = source_model_for_awq
        VLLM_QUANTIZATION = "none"

if VLLM_QUANTIZATION == "awq":
    cfg = dict(awq_recipe_cfg)
    cfg.setdefault("zero_point", True)
    ensure_awq_config_files(Path(VLLM_MODEL_PATH), cfg)

print("Model path for vLLM:", VLLM_MODEL_PATH)
print("vLLM quantisation mode:", VLLM_QUANTIZATION)



## 1B. AWQフェーズ完了後のクリーンアップ

LoRA マージと AWQ 量子化が完了したらこのセルを実行し、AWQ 専用の依存関係をアンインストールしてから vLLM セットアップに進みます。再度 AWQ を行う場合は 1A のセルを再実行してください。

In [ ]:
# === 1B. Clean up AWQ-specific packages ===
pip_uninstall(AWQ_CLEANUP_TARGETS)
pip_install(["setuptools==79.0.1"])
print("AWQ dependencies removed. Ready for vLLM setup.")


## 1C. vLLMフェーズ用依存の再インストール

クリーンアップ後にこのセルを実行して vLLM 関連の依存関係を構築します。必要に応じて `VLLM_SUPPORT_PACKAGES` や `VLLM_CORE_PACKAGES` を調整してください。

In [ ]:
# === 1C. Install dependencies for vLLM ===
pip_install(BASE_BOOTSTRAP)
pip_install(VLLM_SUPPORT_PACKAGES)
pip_install(["compressed-tensors==0.11.0"], extra_args=["--no-deps"])
pip_install(["vllm==0.11.0"], extra_args=["--no-deps"])
print("vLLM dependency stack ready.")

In [ ]:
# === 4. Load vLLM backend helpers ===
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.llmlab.backends.vllm_backend import (
    ModelBundle,
    chat_loop,
    free_model,
    load_model,
    profile_generation,
)

print("vLLM backend utilities loaded.")


In [ ]:
# === 5. Load quantised model into vLLM ===
cfg = {
    "model_name": VLLM_MODEL_PATH,
    "tensor_parallel_size": VLLM_OPTIONS["tensor_parallel_size"],
    "dtype": VLLM_OPTIONS["dtype"],
    "max_model_len": VLLM_OPTIONS["max_model_len"],
    "gpu_memory_utilization": VLLM_OPTIONS["gpu_memory_utilization"],
    "download_dir": str(VLLM_OPTIONS["download_dir"]),
    "quantization": VLLM_QUANTIZATION,
    "lora_path": None,
    "merge_lora": False,
}

print("vLLM load config:", cfg)
bundle: ModelBundle = load_model(cfg)
print("Model loaded. Tokenizer available:", bool(bundle["tok"]))
print("Load timings:", bundle["cfg"].get("_timings", {}))


In [ ]:
# === 6. Batch inference and metrics ===
import pandas as pd

prompts = [
    "Summarise the following specification in Japanese:\n- Multi-stage LoRA tuned Qwen3-14B\n- Target deployment on lightweight edge devices",
    "Explain three benefits of deploying this LoRA-enhanced model as a customer-facing FAQ bot.",
]

outputs, metrics = profile_generation(
    bundle,
    prompts,
    max_new_tokens=GENERATION["MAX_NEW_TOKENS"],
    temperature=GENERATION["TEMPERATURE"],
    top_p=GENERATION["TOP_P"],
    repetition_penalty=GENERATION["REPETITION_PENALTY"],
    stop=GENERATION["STOP"],
)

display(pd.DataFrame({"prompt": prompts, "output": outputs}))
display(pd.DataFrame([metrics]).T.rename(columns={0: "value"}))


In [ ]:
# === 7. Interactive chat (optional) ===
print(f"Chat session started. Type {CHAT['EXIT_COMMAND']} to exit.")
try:
    chat_loop(
        bundle,
        system_prompt=CHAT["SYSTEM_PROMPT"],
        stop_phrases=list({*CHAT["STOP_PHRASES"], CHAT["EXIT_COMMAND"]}),
        max_new_tokens=GENERATION["MAX_NEW_TOKENS"],
        temperature=GENERATION["TEMPERATURE"],
        top_p=GENERATION["TOP_P"],
        repetition_penalty=GENERATION["REPETITION_PENALTY"],
    )
finally:
    print("Chat session finished.")


In [ ]:
# === 8. Cleanup ===
free_model(bundle)
print("Freed vLLM model and tokenizer.")
